# FC, feature joins and validation interfaces

Only the approved continuous 160-frame window is used for FC. Raw structural volumes and ICV are kept separate; cohort-fitted adjustments happen inside training paths. No automatic classification run.

This notebook has NOT been executed on real MRI. Empty paths and unconfirmed recipes intentionally block inference.

In [ ]:
from pathlib import Path
import subprocess, sys
CODE_ROOT = Path('/content/MRI-classification')  # unzip the prepared code package here
assert (CODE_ROOT / 'scripts/run_features.py').is_file(), 'Upload the new local code package first; the current remote branch does not yet contain these scripts.'
PATHS = CODE_ROOT / 'configs/paths.local.yaml'
CONFIG = CODE_ROOT / 'configs/pilot.local.yaml'
HANDOFF = None  # absolute path to the actual QC-approved handoff.json


## Isolated environment
Set CREATE_ENVIRONMENT=True only when you want to install the candidate environment. Weights are acquired separately from the author; no MRI or weight download is performed by these cells.

In [ ]:
CREATE_ENVIRONMENT = False
ENV_DIR = Path('/content/venvs/features')
PYTHON = ENV_DIR / 'bin/python'
if CREATE_ENVIRONMENT:
    if not PYTHON.exists():
        subprocess.run([sys.executable, '-m', 'venv', str(ENV_DIR)], check=True)
    subprocess.run([str(PYTHON), '-m', 'pip', 'install', '-r', str(CODE_ROOT / 'envs/core.txt')], check=True)
assert PYTHON.is_file(), 'Prepare/select the independent environment and set PYTHON.'


## Feature checks and merging
Compute FC from the SAME continuous window as Brain-JEPA. Demographic flags remain pending until real review. Read docs/RUNBOOK.md for structure/merge commands. Do not report accuracy for 3-5 pilot cases.

In [ ]:
assert PATHS.is_file() and CONFIG.is_file() and HANDOFF is not None
subprocess.run([str(PYTHON), str(CODE_ROOT/'scripts/run_features.py'), 'fc', '--paths', str(PATHS), '--config', str(CONFIG), '--handoff', str(HANDOFF)], cwd=CODE_ROOT, check=True)


## Validation interface
D/S/Z/F/U/R are implemented in mri_pipeline.models. Formal evaluation requires confirmed identities/diagnosis timing/QC, training_ready=True, and a separately confirmed fold/tuning configuration. This notebook does not start nested CV.

In [ ]:
sys.path.insert(0, str(CODE_ROOT/'src'))
# If using a separate core interpreter, invoke scripts/evaluate.py there only
# after passing the formal eligibility gate and confirming the fold plan.
print('Pilot goal: real QC, feature alignment, runtime and persistence evidence.')
